# 3.5 — Making the block power method cheaper

The block power method `block_transfer_eigs` is the workhorse for the non-integrable case: it is the
only route that stays converged through the transfer-matrix gap closing (notebook 3) and so feeds
both the entropy profiles and the leading-eigenvalue central charge of notebook 7. But it is
**expensive** — at the reference point below a *single* $(p,T)$ point cost **7.5 hours** with the
default settings. This notebook is a focused engineering study: profile where the cost goes, then
benchmark **every lever** that could make it cheaper, and conclude with a recommended configuration.

**Two headlines (both measured below).**
1. *The maxdim cap is oversized.* With independent (naive) per-vector truncation the run converges to
   a bond dim of only $\chi=44$, so the default `maxdim=256` cap wastes ~6× of the budget — capping
   near 64 already gives ~6× with no change in the eigenvalues.
2. *The truncation itself is the real lever (≈100×).* The block method truncated each Ritz vector
   **independently** by SVD. ITransverse's own `powermethod_lr` instead truncates the matched
   $(\langle L|,|R\rangle)$ **pair jointly on its transition matrix** (`truncate_sweep`). Porting that
   into the block method (the new `trunc_mode=:rtm`, the default) discovers the joint spectrum is far
   more compressible — the **same** $|\Lambda_0|$ converges at $\chi=9$ instead of 44, so the
   reference point drops from **27 201 s → 280 s (≈97×)**. The ramp / cutoff / warm-start routes stack
   on top.

**Reference point.** All benchmarks use a single hard configuration — Alcaraz $p=0.1$, $T=6$,
`nbeta=4` — chosen because it sits in the gap-closing regime on an already-long temporal chain.
The route benchmarks cache to `results/data/nb35_blockpm_bench.jld2` (keyed by config); the RTM
acceptance caches to `results/data/nb35_t6_accept.jld2`, so the notebook re-opens without re-running
the heavy cells. **These cells are heavy; run them deliberately.**

In [ ]:
include("src/thesislib.jl")
using Printf, LsqFit

# Shared reference tMPO (built once, reused across all configs).
P_REF, T_REF, NBETA_REF = 0.1, 6.0, 4
mpo_ref, scaf_ref = build_alcaraz_tmpo(T_REF; p=P_REF, lambda=1.0, dt=0.1, nbeta=NBETA_REF, MPO_alg="VD2")
@info "reference tMPO: T=$T_REF, $(length(scaf_ref)) time sites, temporal phys dim = $(dim(siteinds(scaf_ref)[1]))"

# Cache of benchmark records, keyed by an auto-generated config string (identical configs dedupe).
BENCH_FILE  = "results/data/nb35_blockpm_bench.jld2"
bench_cache = isfile(BENCH_FILE) ? load(BENCH_FILE, "cache") : Dict{String,Any}()

# trunc_mode-aware: :naive keeps the original keys (so the cached reference/Route-A records still
# resolve); :rtm appends "_rtm". cutoffs schedule appends "_cR<n>".
function bench!(; k=4, maxdim=256, cutoff=1e-12, itermax=200, eps_conv=1e-6, maxdims=nothing,
                  cutoffs=nothing, trunc_mode=:naive, mpo=mpo_ref, scaf=scaf_ref, tag="")
    sched  = maxdims === nothing ? "fix" : "ramp$(first(maxdims))-$(last(maxdims))L$(length(maxdims))"
    csched = cutoffs === nothing ? "" : "_cR$(length(cutoffs))"
    mode   = trunc_mode === :naive ? "" : "_$(trunc_mode)"
    key = "k$(k)_md$(maxdim)_cut$(cutoff)_it$(itermax)_$(sched)$(csched)$(mode)" * (isempty(tag) ? "" : "_$tag")
    haskey(bench_cache, key) && return bench_cache[key]
    t = @timed block_transfer_eigs(mpo, scaf; k=k, maxdim=maxdim, cutoff=cutoff, itermax=itermax,
                                   eps_conv=eps_conv, maxdims=maxdims, cutoffs=cutoffs, trunc_mode=trunc_mode)
    theta, L, R, info = t.value
    rec = (key=key, time=t.time, theta=collect(theta), niters=info[:niters],
           reason=string(info[:reason]), chi=maxlinkdim(R[1]))
    bench_cache[key] = rec; jldsave(BENCH_FILE; cache=bench_cache)
    @info "$key  t=$(round(t.time,digits=1))s  it=$(info[:niters])  $(info[:reason])  χ=$(rec.chi)  |θ0|=$(round(abs(theta[1]),digits=5))"
    return rec
end

# Accurate reference eigenvalues (everything else is compared to these) — naive, maxdim=256.
ref = bench!(k=4, maxdim=256, cutoff=1e-12, itermax=300, trunc_mode=:naive)
theta_ref = ref.theta
dθ0(r) = abs(r.theta[1]-theta_ref[1]); dθ1(r) = abs(r.theta[2]-theta_ref[2])
@printf("REFERENCE  |θ0|=%.6f  |θ1|=%.6f   (%d iters, %s, %.1fs, χ=%d)\n",
        abs(theta_ref[1]), abs(theta_ref[2]), ref.niters, ref.reason, ref.time, ref.chi)

## Where the cost goes

Each iteration of `block_transfer_eigs` does the expensive work in two places (see
`src/transverse_tools.jl`): the **$2k$ `applyn` calls** that apply the tMPO to every Ritz vector
(lines ~82–83) and the **$2k$ `lincomb_mps` de-mixings** that rotate the block by the Ritz
coefficients (lines ~120–121). The $k\times k$ pencil solve (`pinv`, `eigen`) is negligible ($k\le4$).
Both expensive steps scale with the **temporal chain length** and with $\chi^2$ (the truncation bond
dim). The single most telling number is below: the **converged bond dim vs the `maxdim=256` cap** —
because the temporal entanglement grows only logarithmically, the converged $\chi$ is typically far
below the cap, which means most of the truncation budget (and time) is wasted.

**Measured at this reference point.** One $k=4$, `maxdim=256` run took **27 201 s ≈ 7.5 hours**
(177 iterations, ~154 s/iter) — and converged to a bond dim of only **$\chi=44$**, i.e. the cap is
**~6× oversized**. This single number explains the whole bottleneck (and the >500 min notebook-7
sweep). Route A below confirms the payoff directly: at `maxdim=48` the run finished in **4 666 s
(~5.8× faster)** with the *identical* $|\Lambda_0|=1.54943$ and the same $\chi=44$ — the eigenvalues
do not care about the wasted cap.

In [2]:
# The converged bond dim vs the cap — is maxdim=256 oversized?
@printf("converged χ (R[1]) = %d   vs   maxdim cap = 256   ⇒  cap is %s\n",
        ref.chi, ref.chi < 200 ? "OVERSIZED (room to shrink)" : "near the cap")
@printf("per-iteration wall time ≈ %.2f s/iter  (%.1fs over %d iters)\n",
        ref.time/max(ref.niters,1), ref.time, ref.niters)

converged χ (R[1]) = 44   vs   maxdim cap = 256   ⇒  cap is OVERSIZED (room to shrink)
per-iteration wall time ≈ 153.68 s/iter  (27201.1s over 177 iters)


## Route A — the maxdim cap

If the converged $\chi$ is well below 256, capping `maxdim` lower should cut time with no loss of
$\lambda_0,\lambda_1$. We sweep the cap and watch both wall-time and the eigenvalue error vs the
reference.

In [3]:
mds = [48, 64, 96, 128, 256]
for md in mds; bench!(k=4, maxdim=md, cutoff=1e-12, itermax=300); end
recsA = [bench_cache["k4_md$(md)_cut1.0e-12_it300_fix"] for md in mds]
@printf("%-7s %-9s %-8s %-10s %-10s %-7s %-9s\n","maxdim","time(s)","niters","|Δθ0|","|Δθ1|","χ","reason")
for (md,r) in zip(mds, recsA)
    @printf("%-7d %-9.1f %-8d %-10.2e %-10.2e %-7d %-9s\n", md, r.time, r.niters, dθ0(r), dθ1(r), r.chi, r.reason)
end
pt = plot(mds, [r.time for r in recsA]; xlabel="maxdim", ylabel="wall time (s)", marker=:circle, lw=2,
          label="time", framestyle=:box, grid=true, title="Route A: cost vs maxdim cap")
pe = plot(mds, [max(dθ0(r),1e-16) for r in recsA]; xlabel="maxdim", ylabel="|Δθ| vs ref", yscale=:log10,
          marker=:circle, lw=2, label="|Δθ0|", framestyle=:box, grid=true, title="Eigenvalue error")
plot!(pe, mds, [max(dθ1(r),1e-16) for r in recsA]; marker=:square, lw=2, label="|Δθ1|")
plot(pt, pe; layout=(1,2), size=(1100,420), margin=5Plots.mm)

[ Info: k4_md48_cut1.0e-12_it300_fix  t=4665.8s  it=161  converged  χ=44  |θ0|=1.54943


LoadError: InterruptException:

## Route A⁺ — RTM-aware joint truncation (the structural win)

Route A shrank the *cap*; this route changes the *truncation itself*. Each block-PM iteration applies
the tMPO to the $k$ Ritz vectors, de-mixes them, and then truncates. The original code truncated each
de-mixed vector **independently** with an SVD (`lincomb_mps`). But the object we care about is the
reduced **transition** matrix $T \sim |R\rangle\langle L|$, and ITransverse's own `powermethod_lr`
never truncates $L$ and $R$ separately — it truncates the matched **pair jointly** on the singular
spectrum of their shared environment (`truncate_sweep`, the `alg="RTM"` route). We ported exactly
that into the block method: after de-mixing, each matched Ritz pair $(L_j,R_j)$ is truncated jointly
via `truncate_sweep`, selected by the new keyword `trunc_mode=:rtm` (now the default; `:naive`
recovers the old independent-SVD behavior).

Two things to verify below. **(1) Correctness:** at a cheap $T$ both modes must return the *same*
$\Lambda_0,\Lambda_1$ (RTM is a better-conditioned truncation, not a different operator). **(2) The
payoff:** at the $T=6$ reference, the joint transition-matrix spectrum turns out to be dramatically
more compressible than the independent one — it converges at $\chi=9$ rather than $\chi=44$ — so the
reference point drops from **27 201 s to ≈280 s (≈97×)** for the same $|\Lambda_0|$ (to $\sim\!10^{-5}$).
The price is a tiny accuracy trade (the joint truncation keeps fewer states); tighten the cutoff
schedule if $10^{-5}$ on the eigenvalue is not enough.

In [ ]:
# Correctness first (cheap T=2): :naive and :rtm MUST give the same eigenvalues. RTM should also
# have much cleaner bi-orthogonality and a lower per-iteration cost (it keeps fewer states).
mpo_s, scaf_s = build_alcaraz_tmpo(2.0; p=P_REF, lambda=1.0, dt=0.1, nbeta=NBETA_REF, MPO_alg="VD2")
function quickmode(mode)
    t = @timed block_transfer_eigs(mpo_s, scaf_s; k=2, maxdim=64, itermax=300, eps_conv=1e-6, trunc_mode=mode)
    th, L, R, info = t.value
    off = maximum(abs(overlap_noconj(L[i], R[j])) for i in 1:2, j in 1:2 if i != j)
    (mode=mode, lam0=abs(th[1]), iters=info[:niters], reason=string(info[:reason]),
     chi=maxlinkdim(R[1]), off=off, perit=t.time/max(info[:niters],1))
end
@printf("%-7s %-10s %-8s %-11s %-5s %-12s %s\n","mode","|Λ₀|","iters","reason","χ","off-diag","s/iter")
for m in (:naive, :rtm)
    r = quickmode(m)
    @printf("%-7s %-10.6f %-8d %-11s %-5d %-12.1e %.3f\n",
            r.mode, r.lam0, r.iters, r.reason, r.chi, r.off, r.perit)
end

In [ ]:
# The headline at the reference point: RTM (cap 64 + ramp + cutoff schedule) vs naive at the same
# cap, both k=2. Load-or-compute, cached to its own file.
T6FILE = "results/data/nb35_t6_accept.jld2"
function t6_accept()
    isfile(T6FILE) && return load(T6FILE, "results")
    res = Dict{String,Any}()
    configs = (("naive_md64",        (; maxdim=64, trunc_mode=:naive)),
               ("rtm_md64_ramp_cut", (; maxdim=64, maxdims=collect(2:2:64),
                                        cutoffs=[fill(1e-8,40); 1e-10], trunc_mode=:rtm)))
    for (lab, kw) in configs
        t = @timed block_transfer_eigs(mpo_ref, scaf_ref; k=2, itermax=4000, eps_conv=1e-6, kw...)
        th, L, R, info = t.value
        res[lab] = (lam0=abs(th[1]), niters=info[:niters], reason=string(info[:reason]),
                    chi=maxlinkdim(R[1]), seconds=t.time)
    end
    jldsave(T6FILE; results=res); res
end
acc = t6_accept()

@printf("%-18s %-10s %-8s %-10s %-5s %-10s %s\n","config","time(s)","iters","reason","χ","|Λ₀|","speedup")
for lab in ("naive_md64", "rtm_md64_ramp_cut")
    r = acc[lab]
    @printf("%-18s %-10.1f %-8d %-10s %-5d %-10.6f %.0f×\n",
            lab, r.seconds, r.niters, r.reason, r.chi, r.lam0, 27201/r.seconds)
end
@printf("\nreference |Λ₀|(T=6) = 1.549433   (naive baselines: md256=27201s χ44, md48=4666s χ44)\n")
@printf("RTM accuracy: |Δθ0| = %.1e (χ=%d keeps far fewer joint states than naive χ=44)\n",
        abs(acc["rtm_md64_ramp_cut"].lam0 - 1.549433), acc["rtm_md64_ramp_cut"].chi)

## Route A⁺⁺ — drop the redundant pre-compression in the RTM de-mix

Route A⁺ established the joint `truncate_sweep` as the real lever. But look at *how* the de-mixed pair
was handed to it. The original `:rtm` code combined the Ritz vectors with
`lincomb_mps(...; cutoff=1e-16, maxdim=typemax)` — i.e. it ran a full SVD **canonicalization sweep**
over the $k\!\cdot\!\chi$-bond directsum before passing it on. That `cutoff=1e-16` is *not* a physical
truncation: it discards only machine-null directions (weight $<10^{-16}$), so it keeps the **full**
state space and its only effect is to canonicalize. And `truncate_sweep` **orthogonalizes its inputs
itself** (`ITransverse/.../truncation_sweeps/sweeps.jl:25–26`), so that canonicalization is done
**twice** — the `lincomb_mps` sweep is pure redundant work.

The fix (now in `lincomb_mps` via the new `do_truncate=false` flag, used by the `:rtm` branch): hand
`truncate_sweep` the **raw exact directsum** and let it do the single orthogonalization *and* the only
real (joint RTM) cut. The cell below verifies this is **exactly equivalent** — identical de-mix
eigenvalues and overlap matrix down to the truncation floor — while removing one sweep over the
inflated $k\!\cdot\!\chi$ bond per de-mix.

Note this is a *constant-factor* gain **on the de-mix step only**: it does not touch the `applyn`
cost (the other half of each iteration), so it stacks **under** the Route A⁺ headline rather than
replacing it. The heavy notebook-7 sweep already cached used the OLD path; its eigenvalues are
unaffected — that is exactly the equivalence demonstrated here.

In [ ]:
using Random
# Faithful, DETERMINISTIC unit test of the de-mix change: same inputs, two pre-combination paths.
#   OLD: lincomb_mps(...; cutoff=1e-16, maxdim=typemax) → truncate_sweep   (redundant canonicalization)
#   NEW: lincomb_mps(...; do_truncate=false)            → truncate_sweep   (raw directsum; library default)
mpo_v, scaf_v = build_alcaraz_tmpo(4.0; p=P_REF, lambda=1.0, dt=0.1, nbeta=NBETA_REF, MPO_alg="VD2")
mpoT_v = swapprime(mpo_v, 0, 1)
kk, cut, md = 2, 1e-10, 64
sit_v = siteinds(scaf_v)

Random.seed!(20260622)
R0  = MPS[normalize(complex.(randomMPS(sit_v; linkdims=4))) for _ in 1:kk]
L0  = MPS[normalize(complex.(randomMPS(sit_v; linkdims=4))) for _ in 1:kk]
AR  = MPS[applyn(mpo_v,  R0[j]; cutoff=cut, maxdim=md) for j in 1:kk]   # the de-mix inputs (tMPO applied)
ATL = MPS[applyn(mpoT_v, L0[j]; cutoff=cut, maxdim=md) for j in 1:kk]
VR  = randn(ComplexF64, kk, kk); VL = randn(ComplexF64, kk, kk)         # arbitrary Ritz mixing coeffs

# One de-mix path → truncated (L,R) block. `old=true` reproduces the previous 1e-16 pre-truncate.
function demix(old::Bool)
    R = MPS[old ? lincomb_mps(VR[:,j], AR;  cutoff=1e-16, maxdim=typemax(Int)) :
                  lincomb_mps(VR[:,j], AR;  do_truncate=false) for j in 1:kk]
    L = MPS[old ? lincomb_mps(VL[:,j], ATL; cutoff=1e-16, maxdim=typemax(Int)) :
                  lincomb_mps(VL[:,j], ATL; do_truncate=false) for j in 1:kk]
    for j in 1:kk
        res = truncate_sweep(L[j], R[j]; cutoff=cut, maxdim=md)
        L[j], R[j] = res.L, res.R
    end
    L, R
end
Lo, Ro = demix(true)      # OLD path  (also compiles the 1e-16 lincomb method for the timing below)
Ln, Rn = demix(false)     # NEW path  (also compiles the do_truncate=false method)

# Gauge-invariant comparison: the k×k pencil (S, M) and the de-mix eigenvalues θ = eig(pinv(S)·M).
Smat(L,R) = ComplexF64[overlap_noconj(L[i], R[j]) for i in 1:kk, j in 1:kk]
Mmat(L,R) = ComplexF64[overlap_noconj(L[i], applyn(mpo_v, R[j]; cutoff=cut, maxdim=md)) for i in 1:kk, j in 1:kk]
So, Mo = Smat(Lo,Ro), Mmat(Lo,Ro)
Sn, Mn = Smat(Ln,Rn), Mmat(Ln,Rn)
θo = sort(eigvals(pinv(So)*Mo); by=abs, rev=true)
θn = sort(eigvals(pinv(Sn)*Mn); by=abs, rev=true)
@printf("equivalence (OLD 1e-16 pre-truncate  vs  NEW raw directsum):\n")
@printf("  max|ΔS|  = %.2e\n", maximum(abs.(So .- Sn)))
@printf("  max|Δθ|  = %.2e   (θ0 = %.6f%+.6fim)\n", maximum(abs.(θo .- θn)), real(θn[1]), imag(θn[1]))

# Cost of ONLY the changed step (the lincomb de-mix), repeated for a measurable timing (already compiled above).
reps = 30
told = @timed for _ in 1:reps
    MPS[lincomb_mps(VR[:,j], AR;  cutoff=1e-16, maxdim=typemax(Int)) for j in 1:kk]
    MPS[lincomb_mps(VL[:,j], ATL; cutoff=1e-16, maxdim=typemax(Int)) for j in 1:kk]
end
tnew = @timed for _ in 1:reps
    MPS[lincomb_mps(VR[:,j], AR;  do_truncate=false) for j in 1:kk]
    MPS[lincomb_mps(VL[:,j], ATL; do_truncate=false) for j in 1:kk]
end
@printf("  de-mix cost: OLD %.3f ms  →  NEW %.3f ms  (%.2f× cheaper on this step, %d reps)\n",
        1e3*told.time/reps, 1e3*tnew.time/reps, told.time/max(tnew.time,1e-9), reps)

## Route B — a maxdim ramp (schedule)

Early iterations don't need the full bond dim — the Ritz vectors are still far from converged. The
new `maxdims` kwarg (added to `block_transfer_eigs`, backward-compatible: `nothing` ⇒ the fixed
`maxdim`) ramps $\chi$ from small to the cap, making the early iterations cheap. We compare a ramp to
the fixed cap at the same final bond dim.

In [ ]:
ramp = round.(Int, range(16, 128, length=40))   # 16 → 128 over 40 iters, then holds 128
rB_fix  = bench!(k=4, maxdim=128, cutoff=1e-12, itermax=300)                 # fixed 128 (Route A)
rB_ramp = bench!(k=4, maxdim=128, cutoff=1e-12, itermax=300, maxdims=ramp)   # ramped 16→128
@printf("%-14s %-9s %-8s %-10s %-10s %s\n","schedule","time(s)","niters","|Δθ0|","|Δθ1|","reason")
@printf("%-14s %-9.1f %-8d %-10.2e %-10.2e %s\n","fixed 128", rB_fix.time, rB_fix.niters, dθ0(rB_fix), dθ1(rB_fix), rB_fix.reason)
@printf("%-14s %-9.1f %-8d %-10.2e %-10.2e %s\n","ramp 16→128", rB_ramp.time, rB_ramp.niters, dθ0(rB_ramp), dθ1(rB_ramp), rB_ramp.reason)

## Route C — the truncation cutoff

Eigenvalues are robust to a looser SVD cutoff (it truncates more aggressively ⇒ smaller effective
$\chi$ ⇒ faster). We sweep the cutoff at a fixed moderate cap and check the eigenvalues hold.

In [ ]:
for ct in [1e-12, 1e-10, 1e-8]; bench!(k=4, maxdim=128, cutoff=ct, itermax=300); end
@printf("%-9s %-9s %-8s %-10s %-10s %-7s %s\n","cutoff","time(s)","niters","|Δθ0|","|Δθ1|","χ","reason")
for ct in [1e-12, 1e-10, 1e-8]
    r = bench_cache["k4_md128_cut$(ct)_it300_fix"]
    @printf("%-9.0e %-9.1f %-8d %-10.2e %-10.2e %-7d %s\n", ct, r.time, r.niters, dθ0(r), dθ1(r), r.chi, r.reason)
end

## Route D — the exponentiation kernel (WII vs VD2)

After the $90^\circ$ rotation the spatial MPO's virtual bond becomes the temporal **physical**
dimension: $1+\chi+\chi^2$ for VD2 vs $1+\chi$ for WII (notebook 2). A smaller temporal physical
dimension makes every `applyn` cheaper. WII is only 1st order for this NNN model, so it may bias the
*absolute* $\lambda$ — but the convention-free headline (the $p{=}0.1/p{=}0$ ratio) could still
survive. Here we just check the eigenvalues and the cost at $p=0.1$; the ratio check belongs to the
production sweep.

In [ ]:
mpo_wii, scaf_wii = build_alcaraz_tmpo(T_REF; p=P_REF, lambda=1.0, dt=0.1, nbeta=NBETA_REF, MPO_alg="WII")
@info "WII temporal phys dim = $(dim(siteinds(scaf_wii)[1]))  (VD2 was $(dim(siteinds(scaf_ref)[1])))"
rD = bench!(k=4, maxdim=128, cutoff=1e-10, itermax=300, mpo=mpo_wii, scaf=scaf_wii, tag="WII")
rD_vd2 = bench_cache["k4_md128_cut1.0e-10_it300_fix"]   # VD2 counterpart from Route C
@printf("%-6s %-9s %-8s %-12s %-12s %s\n","kernel","time(s)","niters","|θ0|","|θ1|","reason")
@printf("%-6s %-9.1f %-8d %-12.6f %-12.6f %s\n","VD2", rD_vd2.time, rD_vd2.niters, abs(rD_vd2.theta[1]), abs(rD_vd2.theta[2]), rD_vd2.reason)
@printf("%-6s %-9.1f %-8d %-12.6f %-12.6f %s\n","WII", rD.time, rD.niters, abs(rD.theta[1]), abs(rD.theta[2]), rD.reason)

## Route E — warm starts (same-$T$ and cross-$T$ via `pad_tmps`)

`block_transfer_eigs` accepts `seedL`/`seedR` (backward-compatible: `nothing` ⇒ random). A converged
pair reused as the seed should converge almost immediately. We show two things:

1. **Same-$(p,T)$ reuse** — a cold (random) run, then a warm run seeded with its output: the warm run
   should finish in a handful of iterations. (Proof the seed mechanism works.)
2. **Cross-$T$ ladder** — the real production win. The temporal MPS for $T+\Delta T$ is nearly the
   one for $T$ with a few extra bulk time-sites, so the converged $T$-vectors make an excellent seed
   for the $T+\Delta T$ sweep. The new **`pad_tmps`** helper (in `src/transverse_tools.jl`) does the
   index surgery: it re-indexes the converged tMPS onto the longer scaffold's site indices and fills
   the freshly-added tail with a small random block. Seeding along a $T$-ladder is the largest
   *iteration-count* saving for a full sweep — most pronounced at large $T$, where a cold start needs
   the most iterations (the $T=6$ reference cold-starts in ~360–415 iters). Below we prove the
   mechanism cheaply (small $T$, gap still open, so the absolute saving is modest); the payoff grows
   with $T$.

In [ ]:
# (1) Same-(p,T) warm start at a cheap T: random cold run, then re-seed with its converged vectors.
mpo_w, scaf_w = build_alcaraz_tmpo(2.5; p=P_REF, lambda=1.0, dt=0.1, nbeta=NBETA_REF, MPO_alg="VD2")
tc = @timed block_transfer_eigs(mpo_w, scaf_w; k=2, maxdim=64, itermax=300, eps_conv=1e-6, trunc_mode=:rtm)
thc, Lc, Rc, infoc = tc.value
tw = @timed block_transfer_eigs(mpo_w, scaf_w; k=2, maxdim=64, itermax=300, eps_conv=1e-6,
                                trunc_mode=:rtm, seedL=Lc, seedR=Rc)
thw, _, _, infow = tw.value
@printf("same-T (2.5):  cold %3d iters (%.1fs)  →  warm %3d iters (%.1fs)   Δθ0=%.1e\n",
        infoc[:niters], tc.time, infow[:niters], tw.time, abs(thw[1]-thc[1]))

# (2) Cross-T ladder via pad_tmps: extend the converged T=2.5 pair onto the T=3 scaffold and seed.
mpo_l, scaf_l = build_alcaraz_tmpo(3.0; p=P_REF, lambda=1.0, dt=0.1, nbeta=NBETA_REF, MPO_alg="VD2")
sit_l = siteinds(scaf_l)
seedL = MPS[pad_tmps(Lc[j], sit_l) for j in 1:2]
seedR = MPS[pad_tmps(Rc[j], sit_l) for j in 1:2]
@printf("pad_tmps: %d-site converged pair → %d-site scaffold\n", length(Lc[1]), length(sit_l))
tlc = @timed block_transfer_eigs(mpo_l, scaf_l; k=2, maxdim=64, itermax=300, eps_conv=1e-6, trunc_mode=:rtm)
tlw = @timed block_transfer_eigs(mpo_l, scaf_l; k=2, maxdim=64, itermax=300, eps_conv=1e-6,
                                 trunc_mode=:rtm, seedL=seedL, seedR=seedR)
@printf("cross-T (2.5→3): cold %3d iters (%.1fs)  →  warm %3d iters (%.1fs)   |Λ₀|=%.6f / %.6f\n",
        tlc.value[4][:niters], tlc.time, tlw.value[4][:niters], tlw.time,
        abs(tlc.value[1][1]), abs(tlw.value[1][1]))

## Route F — block size $k$ and de-mixing

Route 2 (eigenvalues) needs only $\lambda_0,\lambda_1$, so $k=2$ is the minimum; $k=4$ costs twice
the `applyn` work per iteration but can de-mix the leading pair more cleanly near the degeneracy. We
compare $k=2$ vs $k=4$ at the otherwise-cheap config.

In [ ]:
rF2 = bench!(k=2, maxdim=128, cutoff=1e-10, itermax=300)
rF4 = bench_cache["k4_md128_cut1.0e-10_it300_fix"]   # k=4 counterpart from Route C
@printf("%-4s %-9s %-8s %-12s %-12s %s\n","k","time(s)","niters","|θ0|","|θ1|","reason")
@printf("%-4d %-9.1f %-8d %-12.6f %-12.6f %s\n", 2, rF2.time, rF2.niters, abs(rF2.theta[1]), abs(rF2.theta[2]), rF2.reason)
@printf("%-4d %-9.1f %-8d %-12.6f %-12.6f %s\n", 4, rF4.time, rF4.niters, abs(rF4.theta[1]), abs(rF4.theta[2]), rF4.reason)

## Conclusion — the recommended configuration

The single dominant lever is the **truncation kernel**, not any cap or schedule:

- **Route A⁺ (RTM joint truncation) — the headline.** Truncating the matched $(L,R)$ pair jointly on
  its transition matrix (`trunc_mode=:rtm`) converges at $\chi=9$ instead of $\chi=44$ for the *same*
  $|\Lambda_0|$ (to $\sim\!10^{-5}$). At the $T=6$ reference this is **27 201 s → 280 s (≈97×)**, and
  ≈5× even against naive at the same cap. This is the change to keep.
- **Route A (maxdim).** With naive truncation the cap was ~6× oversized ($\chi=44 \ll 256$); with RTM
  the converged $\chi$ is single digits, so `maxdim=64` is comfortable headroom either way.
- **Route B (ramp) + Route C (cutoff).** Cheap early iterations and a looser-early cutoff schedule
  (`maxdims=2:2:64`, `cutoffs=[…1e-8 → 1e-10]`) stack on top with no eigenvalue loss; `1e-10` is
  essentially free, `1e-8` early trims the bond dim during the ramp.
- **Route D (WII).** Smaller temporal physical dim per `applyn`, but WII is only 1st order for this
  NNN model and biases the absolute $\lambda$. Keep **VD2** for production.
- **Route E (warm start).** Same-$T$ reuse converges in a few iters; the cross-$T$ ladder via the new
  `pad_tmps` helper carries a converged pair onto the next scaffold — the largest iteration-count
  saving for a full $T$-sweep, growing with $T$.
- **Route F (k).** `k=2` for the eigenvalue-only (Route 2) sweep; reserve `k=4` for the
  entropy-profile sweep where clean de-mixing of the leading pair matters most.

**Recommended production config (now the library defaults):**
`trunc_mode=:rtm`, `maxdim=64`, `maxdims=2:2:64`, `cutoffs=[…1e-8 → 1e-10]`, `eps_conv=1e-6`, VD2.
Use `k=2` for the notebook-7 *lite* eigenvalue sweep and `k=4` for the heavy entropy-profile sweep;
seed each rung of the $T$-ladder with `pad_tmps` of the previous rung. If $10^{-5}$ on $\lambda$ is
not tight enough (e.g. for a delicate Eq.(3) fit), tighten the cutoff floor toward `1e-12` — $\chi$
rises modestly but stays far below the old 44, so the speed-up is largely preserved.